In [ ]:
#!/usr/bin/env python3
"""
Project 13 — G0b (b/3): masked re-run of Path A Part 1's "immune = highest
decodability" lineage result, excluding VDJ/Ig/TCR/HLA loci (src/24's
data/processed/vdj_hla_mask.csv). Run on Kaggle (docs/TASK_v3_G0b_vdj_hla_audit.md).

WHY THIS COULDN'T RUN IN THE DEV SANDBOX
------------------------------------------
Part 1's headline number (obs['test_pearson'] in metadata.h5ad) is a SCALAR
Decima ships per pseudobulk -- its own held-out-gene Pearson correlation
against its own training targets. metadata.h5ad's `layers` only contains
prediction replicates (preds, v1_rep0-3); there is no ground-truth
expression layer in that file, so test_pearson cannot be decomposed
gene-by-gene and re-excluded for 45 specific loci without the actual
per-gene training targets Decima evaluated against. Confirmed by direct
inspection (`h5py` on data/raw/metadata.h5ad in-sandbox): no `X`, no
`layers['targets']` or equivalent.

WHAT THIS SCRIPT DOES INSTEAD
-------------------------------
Fetches Decima's real per-gene training/eval targets (the actual measured
pseudobulk expression Decima was trained/evaluated on -- NOT metadata.h5ad,
a separate, larger file/dataset also published under Genentech/decima-data
on HuggingFace; see `decima` package docs / `decima.utils.load_dataset` for
the canonical loader) and recomputes, per pseudobulk instance, a held-out
gene-set Pearson correlation TWICE: once over all measured genes (should
closely reproduce the shipped test_pearson, sanity check), once with the
45 VDJ_HLA_MASK genes additionally excluded from the correlation. Then
re-runs Part 1's exact confound-control (OLS vs log depth/n_genes/study FE)
and lineage-means aggregation on both, and reports the before/after
lineage ranking -- specifically whether "immune" stays the highest-mean
lineage.

If `decima`'s real target data turns out to be inaccessible or too large
even for Kaggle within a session, fall back documented at the bottom
(FALLBACK section) -- report that honestly rather than faking a result.

Usage
-----
    On Kaggle: New Notebook -> enable Internet -> paste/upload this file ->
    Run All. Needs: `pip install decima huggingface_hub h5py pandas numpy
    scipy statsmodels`. Mirrors kaggle_pathA_part1_decodability_map.py's
    path conventions (IS_KAGGLE branch -> /kaggle/working).

Output
------
    data/processed/gate0b_pathA_lineage_before_after_report.json
    data/processed/atlas_celltype_decodability_map_masked.parquet
"""
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd

IS_KAGGLE = Path("/kaggle").exists()
REPO = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent.parent
RAW = REPO / "data" / "raw"
PROC = REPO / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

METADATA_H5AD = RAW / "metadata.h5ad"
MASK_CSV = PROC / "vdj_hla_mask.csv"
IN_CELLTYPE_MAP = PROC / "atlas_celltype_decodability_map.parquet"   # Part 1's original (unmasked) output
OUT_REPORT = PROC / "gate0b_pathA_lineage_before_after_report.json"
OUT_MAP_MASKED = PROC / "atlas_celltype_decodability_map_masked.parquet"


def log(m):
    print(f"[g0b-pathA] {m}", flush=True)


def load_mask_gene_ids():
    if not MASK_CSV.exists():
        sys.exit(f"[error] need {MASK_CSV} -- run src/24_gate0b_vdj_hla_mask_v1.py first "
                  f"and copy vdj_hla_mask.csv into this Kaggle session's data/processed/")
    return set(pd.read_csv(MASK_CSV)["ensembl_id"])


def try_load_real_targets():
    """
    Attempt to load Decima's actual per-gene training/eval targets (the
    ground truth test_pearson was computed against). This is the piece
    NOT present in metadata.h5ad. Try the documented `decima` package
    loader first; fall back to a direct HuggingFace file probe.
    """
    try:
        import decima
        log("`decima` package available -- attempting decima.utils dataset loader "
            "(adjust call per whatever version is installed on Kaggle; API surface "
            "may differ from what's assumed here, inspect `dir(decima)`/`dir(decima.utils)` "
            "if this fails)")
        # NOTE: exact loader call intentionally left for the Kaggle session to adapt --
        # different decima package versions expose this differently
        # (e.g. decima.utils.load_dataset, decima.data.AnnDataset, etc.)
        raise NotImplementedError(
            "Fill in the real loader call here once you've inspected `decima`'s API on "
            "Kaggle (`import decima; help(decima)`), OR locate the HuggingFace file that "
            "ships the real training targets (distinct from metadata.h5ad) via "
            "`huggingface_hub.list_repo_files('Genentech/decima-data', repo_type='dataset')` "
            "and adapt this function to load it directly with h5py/anndata."
        )
    except Exception as e:
        log(f"[fallback] real target loader not wired up / failed: {e}")
        return None


def main():
    mask_ids = load_mask_gene_ids()
    log(f"{len(mask_ids)} masked loci loaded")

    targets = try_load_real_targets()
    if targets is None:
        # FALLBACK: cannot recompute test_pearson gene-masked without real targets.
        # Report this honestly instead of faking a number.
        report = {
            "status": "BLOCKED",
            "reason": ("Ground-truth per-gene expression targets for Decima's shipped "
                       "test_pearson are not present in metadata.h5ad (confirmed: only "
                       "prediction layers, no measured/target layer) and this Kaggle "
                       "session's real-target loader is not yet wired up (see "
                       "try_load_real_targets()). Locate Decima's actual training/eval "
                       "target data on HuggingFace (Genentech/decima-data has more files "
                       "than metadata.h5ad -- check via huggingface_hub.list_repo_files) "
                       "or via the `decima` package's own dataset loader, fill in "
                       "try_load_real_targets(), and re-run."),
            "n_masked_loci": len(mask_ids),
            "context": ("Both other G0b legs -- identity-gap candidates (src/25) and the "
                       "LOEUF constraint test (src/26) -- were fully re-derivable from "
                       "data already in hand and showed NEGLIGIBLE movement after masking "
                       "(masked loci were 0.245% of the identity-gap table and an "
                       "unmeasurable fraction of the matched candidate/background sets). "
                       "Given 45/18,457 genes (0.24% of the panel) is the same tiny "
                       "fraction that moved (a) and (c) by <1%, a similarly small effect "
                       "on (b) would be the base-rate expectation -- but this is NOT "
                       "reported as a result until actually run."),
        }
        with open(OUT_REPORT, "w") as f:
            json.dump(report, f, indent=2)
        log("[BLOCKED] wrote report explaining why -- fill in try_load_real_targets() and re-run")
        return

    # ... (recompute path once targets are wired up: mirror
    # kaggle_pathA_part1_decodability_map.py's confound_control() +
    # lineage aggregation, computing test_pearson per pseudobulk twice --
    # all genes vs genes-minus-mask -- then diff the lineage_means dict)
    raise NotImplementedError("targets loader succeeded -- implement the masked "
                               "recompute + before/after diff here")


if __name__ == "__main__":
    main()